# LangChain Fundamentals
### From Raw OpenAI Calls → Composable AI Pipelines

---

## Background and motivation

You already know how to make OpenAI API calls that look something like this:

```python
from openai import OpenAI
client = OpenAI()
response = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": "Tell me a joke"}]
)
print(response.choices[0].message.content)
```

---

## What We'll Cover

| Section | Concept | Why It Matters |
|---------|---------|----------------|
| 1 | Setup & the LangChain model wrapper | Swap models without rewriting logic |
| 2 | Prompt Templates | Reusable, parameterized prompts |
| 3 | LCEL — The `|` pipe operator | Compose steps into readable pipelines |
| 4 | Output Parsers | Get structured data, not raw strings |
| 5 | Chat History & Memory | Stateful multi-turn conversations |
| 6 | Putting It All Together | A complete mini-app |

---

## Core Mental Model

LangChain is built around one idea: **everything is a Runnable**.

A `Runnable` is anything that can receive input and produce output. That means:
- A prompt template is a Runnable
- An LLM is a Runnable
- An output parser is a Runnable
- A chain of all three is also a Runnable

Because they all share the same interface, you can connect them with the `|` pipe operator — just like Unix pipes:

```
input → prompt_template | llm | output_parser → final output
```

This is called **LCEL** (LangChain Expression Language), and it's the foundation of everything you'll use.

In [ ]:
%load_ext dotenv
%dotenv ../../05_src/.env
%dotenv ../../05_src/.secrets

---
## Section 1: The LangChain Model Wrapper

### Why not just use the OpenAI client directly?

You can — but LangChain wraps the model so that:
1. You can **switch providers** (Anthropic, Mistral, Ollama) by changing one line
2. The model becomes a **composable piece** of a larger pipeline
3. You get built-in **retry logic, streaming, and callbacks** for free

### `ChatOpenAI` — the model wrapper

This replaces your manual `OpenAI()` client. Under the hood it still calls the same API.

In [ ]:
import os
from openai import OpenAI
from langchain_openai import ChatOpenAI



llm = ChatOpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
                 model="gpt-4o-mini",   # Which model to use
                 temperature=0.7,        # 0 = deterministic, 1 = creative
                 max_tokens=500,          # Max response length 
                 api_key='any value',
                 default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

print("Model created:", llm.model_name)

In [ ]:
# to disable tracing - by defualt LangChain will try to send telemetry to LangSmith
os.environ["LANGCHAIN_TRACING_V2"] = "false"

In [ ]:
# The simplest possible call: just invoke with a string
# LangChain converts this to a proper message format behind the scenes
response = llm.invoke("What is LangChain in one sentence?")

# response is an AIMessage object, not a raw string
print("Type:", type(response))
print("Content:", response.content)

In [ ]:
# You can also pass structured messages — just like you did with the raw OpenAI client
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage(content="You are a pirate. Respond in Shakesperean style."),
    HumanMessage(content="What is Python programming?")
]

response = llm.invoke(messages)
print(response.content)

---
## Section 2: Prompt Templates

### The Problem With Hardcoded Prompts

In raw OpenAI code you might do:
```python
`prompt = f"Explain {topic} in simple terms for a {audience} audience.`
```

This works, but it's just string formatting. **Prompt Templates** solve this properly:
- They're **reusable and testable** objects
- They can be **shared and versioned**
- They become **composable Runnables** that plug into chains

### Two Types We'll Use

| Template | Use case |
|----------|----------|
| `PromptTemplate` | Single-string prompts (no chat history) |
| `ChatPromptTemplate` | Structured multi-role prompts (System + Human) |

In [ ]:
from langchain_core.prompts import PromptTemplate

# Define a reusable prompt with placeholders in {curly braces}
template = PromptTemplate(
    input_variables=["topic", "audience"],
    template="Explain {topic} in simple terms for a {audience} audience. Use an analogy."
)

# .format() fills in the placeholders and gives you the final string
filled_prompt = template.format(topic="relativity", audience="high school student")
print("Filled prompt:")
print(filled_prompt)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# ChatPromptTemplate lets you define system + human messages with variables
chat_template = ChatPromptTemplate.from_messages([
    ("system", "You are an expert in {domain}. Be concise and use bullet points."),
    ("human", "What are the top 3 things a beginner should know about {topic}?")
])

# .invoke() produces the list of messages ready to send to the LLM
messages = chat_template.invoke({"domain": "public health", "topic": "basic personal care to avoid spreading germs"})

print("Produced messages:")
for msg in messages.messages:
    print(f"  [{msg.type.upper()}]: {msg.content[:80]}...")

---
## Section 3: LCEL — Chaining with the `|` Pipe Operator

### The Key Insight

Every LangChain component — prompts, models, parsers — implements `.invoke()`. This shared interface lets you **pipe them together** using `|`:

```python
chain = prompt | llm | parser
result = chain.invoke({"topic": "..."})
```

This is **LCEL** (LangChain Expression Language). What happens when you call `.invoke()`:
1. The dict is passed into `prompt` → produces a `ChatPromptValue`
2. That value is passed into `llm` → produces an `AIMessage`
3. That message is passed into `parser` → produces whatever type the parser returns

### Why This Is Powerful
- No callbacks, no glue code — just composition
- The entire chain is itself a Runnable (you can pipe it further)
- You get streaming, batching, and async for free on any chain

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# Step 1: Define each component
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that explains complicated concepts clearly."),
    ("human", "Explain {concept} in exactly 2 sentences.")
])


llm = ChatOpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
                 model="gpt-4o-mini",   # Which model to use
                 temperature=0,        # 0 = deterministic, 1 = creative
                 api_key='any value',
                 default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

parser = StrOutputParser()  # Extracts the string from the AIMessage

# Step 2: Build the chain by piping them together
# This does NOT call the API yet — it just wires the components
chain = prompt | llm | parser

print("Chain type:", type(chain))
print("Chain is a Runnable:", hasattr(chain, 'invoke'))

In [ ]:
# Step 3: Run the chain — THIS is when the API is called
result = chain.invoke({"concept": "gradient descent"})

print("Result type:", type(result))  # str — because of StrOutputParser
print()
print(result)

In [ ]:
# .batch() runs the chain on multiple inputs at once (in parallel by default)
# This is much faster than calling .invoke() in a loop

concepts = [
    {"concept": "tokens"},
    {"concept": "attention mechanism"},
    {"concept": "backpropagation"},
]

results = chain.batch(concepts)

for concept, result in zip(concepts, results):
    print(f" {concept['concept']}")
    print(f"   {result}")
    print()

In [ ]:
# .stream() yields tokens as they arrive — useful for chat UIs
print("Streaming response: ", end="")
for chunk in chain.stream({"concept": "vector embeddings"}):
    print(chunk, end="", flush=True)
print()  # newline at the end

---
## Section 4: Output Parsers

### Why Output Parsers?

LLMs return strings. Your application often needs **structured data** — a dict, a list, a Python object. Output parsers bridge that gap.

| Parser | Returns | Best for |
|--------|---------|----------|
| `StrOutputParser` | `str` | Simple text responses |
| `JsonOutputParser` | `dict` | When you want JSON |
| `PydanticOutputParser` | Pydantic model | Type-safe structured output |
| `CommaSeparatedListOutputParser` | `list[str]` | Quick lists |

### The Magic: Format Instructions

Parsers can inject **format instructions** into your prompt automatically. You add `{format_instructions}` as a placeholder and the parser fills it in with instructions telling the LLM exactly how to format its response.

In [ ]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
                 model="gpt-4o-mini",   # Which model to use
                 temperature=0,        # 0 = deterministic, 1 = creative
                 api_key='any value',
                 default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

# JsonOutputParser automatically parses the LLM's JSON response
json_parser = JsonOutputParser()

prompt = ChatPromptTemplate.from_messages([
    ("system", "Respond ONLY with valid JSON. No extra text."),
    ("human", "Give me info about {language} programming language as JSON with keys: name, year_created, creator, main_use_case, difficulty")
])

chain = prompt | llm | json_parser

result = chain.invoke({"language": "Python"})

print("Result type:", type(result))  # dict!
print()
for key, value in result.items():
    print(f"  {key}: {value}")

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import List

# Define EXACTLY what structure you want back
class SecurityThreat(BaseModel):
    name: str = Field(description="Name of the security threat")
    severity: str = Field(description="Severity level: low, medium, high, or critical")
    description: str = Field(description="Brief description of the threat")
    mitigation: str = Field(description="One key mitigation strategy")

class ThreatList(BaseModel):
    threats: List[SecurityThreat] = Field(description="List of security threats")

# Create the parser with your Pydantic model
pydantic_parser = PydanticOutputParser(pydantic_object=ThreatList)

# The parser generates format instructions automatically
# We inject them into the prompt with {format_instructions}
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a cybersecurity expert.\n{format_instructions}"),
    ("human", "List the top 3 threats in {environment} environments.")
]).partial(format_instructions=pydantic_parser.get_format_instructions())

chain = prompt | llm | pydantic_parser

result = chain.invoke({"environment": "cloud"})

print("Result type:", type(result))  # ThreatList Pydantic object!
print()
for threat in result.threats:
    print(f" {threat.name} [{threat.severity.upper()}]")
    print(f"   {threat.description}")
    print(f"   ✅ Mitigation: {threat.mitigation}")
    print()

---
## Section 5: Chat History & Memory

### The Problem: LLMs Are Stateless

Every call to an LLM is independent — it doesn't remember previous messages unless you include them in the prompt. Managing that history manually (keeping a list, appending to it, trimming it) gets complex fast.

LangChain gives you two clean tools:
1. **`ChatMessageHistory`** — stores the conversation as a list of messages
2. **`RunnableWithMessageHistory`** — wraps your chain to automatically read/write history

### How It Works

```
User sends message
    ↓
RunnableWithMessageHistory pulls history from store by session_id
    ↓
Injects history into the prompt
    ↓
Runs the chain
    ↓
Saves the new exchange back to the store
    ↓
Returns the response
```

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
                 model="gpt-4o-mini",   # Which model to use
                 temperature=0.7,        # 0 = deterministic, 1 = creative
                 api_key='any value',
                 default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

# MessagesPlaceholder is a special placeholder that gets replaced
# with the actual list of past messages when the chain runs
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Be concise."),
    MessagesPlaceholder(variable_name="chat_history"),  # ← history goes here
    ("human", "{input}")                                # ← new message goes here
])

# Build the core chain (no memory yet)
chain = prompt | llm | StrOutputParser()

# --- Store for chat histories ---
# In production you'd use Redis or a database
# Here we use a simple in-memory dict keyed by session_id
store = {}  # { session_id: InMemoryChatMessageHistory }

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    """Return the chat history for a session, creating it if needed."""
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# Wrap the chain with memory management
chat_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",         # which key holds the new message
    history_messages_key="chat_history" # which placeholder to fill with history
)

print(" Chat chain with memory ready")

In [ ]:
# Each invocation needs a config with the session_id
# Same session_id = same conversation thread
session_config = {"configurable": {"session_id": "workshop-demo-1"}}

# Turn 1
response1 = chat_with_history.invoke(
    {"input": "My name is Alex and I work in application development."},
    config=session_config
)
print("Turn 1 →", response1)
print()

# Turn 2 — the LLM should remember Alex's name and job
response2 = chat_with_history.invoke(
    {"input": "What topics should I learn about AI given my background?"},
    config=session_config
)
print("Turn 2 →", response2)
print()

# Turn 3 — test memory further
response3 = chat_with_history.invoke(
    {"input": "What was my name again?"},
    config=session_config
)
print("Turn 3 →", response3)

In [ ]:
# Inspect what's stored in memory
history = store["workshop-demo-1"]
print(f"Messages in session: {len(history.messages)}")
print()
for msg in history.messages:
    role = " Human" if msg.type == "human" else " AI"
    print(f"{role}: {msg.content[:100]}..." if len(msg.content) > 100 else f"{role}: {msg.content}")

---
## Section 6: Putting It All Together — Mini App

Let's build a **multi-turn technical Q&A assistant** that:
- Takes a domain/specialization at startup
- Answers questions with structured output (a clear answer + a follow-up question)
- Remembers the entire conversation

This combines everything: `ChatPromptTemplate`, `LCEL`, `PydanticOutputParser`, and `RunnableWithMessageHistory`.

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_openai import ChatOpenAI

# --- Structured Output schema (unchanged) ---
class QAResponse(BaseModel):
    answer: str = Field(description="Clear, concise answer to the question")
    key_takeaway: str = Field(description="The single most important point to remember")
    follow_up: str = Field(description="A thought-provoking follow-up question to deepen understanding")

# --- Model ---
#llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
llm = ChatOpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
                 model="gpt-4o-mini",   # Which model to use
                 temperature=0.3,        # 0 = deterministic, 1 = creative
                 api_key='any value',
                 default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

# Key fix: with_structured_output() uses OpenAI's native function calling
# to guarantee a valid QAResponse object back — no JSON prompt tricks needed.
# This works reliably even with long conversation histories.
structured_llm = llm.with_structured_output(QAResponse)

# --- Prompt — no format_instructions needed anymore ---
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert in {domain}. Teach clearly and build on prior conversation."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}")
])

# --- Chain ---
chain = prompt | structured_llm

# --- Manual history store (unchanged from previous fix) ---
session_store = {}

def get_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in session_store:
        session_store[session_id] = InMemoryChatMessageHistory()
    return session_store[session_id]

# --- Helper (unchanged) ---
def ask(question: str, domain: str = "AI and machine learning", session: str = "main"):
    history = get_history(session)

    response = chain.invoke({
        "question": question,
        "domain": domain,
        "chat_history": history.messages
    })

    history.add_user_message(question)
    history.add_ai_message(response.answer)

    print(f" {question}")
    print(f" Answer: {response.answer}")
    print(f" Key takeaway: {response.key_takeaway}")
    print(f" Follow-up: {response.follow_up}")
    print("-" * 60)
    return response

print(" Assistant ready.")

In [ ]:
# Ask a sequence of related questions — notice how answers build on context
r1 = ask("What is an embedding in the context of LLMs?")
print()
r2 = ask("How do embeddings relate to what you just explained?")  # Tests memory
print()
r3 = ask("Give me a practical use case for this in application development")

---
## Summary & Key Takeaways

| Concept | What It Is | Code |
|---------|-----------|------|
| `ChatOpenAI` | LangChain's wrapper for OpenAI models | `llm = ChatOpenAI(model="gpt-4o-mini")` |
| `ChatPromptTemplate` | Reusable, structured prompt with variables | `from_messages([("system", ...), ("human", ...)])` |
| LCEL `\|` operator | Compose components into a pipeline | `chain = prompt \| llm \| parser` |
| `StrOutputParser` | Extract plain text from LLM response | Appended at end of chain |
| `PydanticOutputParser` | Get type-safe structured output | Define a `BaseModel`, wrap in parser |
| `RunnableWithMessageHistory` | Add conversation memory to any chain | Wraps chain + session store |

### What's Next?

**Tools** and **LangGraph** — making our chains dynamic and capable of taking actions (searching the web, reading files, calling APIs) based on what the LLM decides to do.